**Investments: Theory and Data Analysis**, Bates, Boyer, and Fletcher

# Chapter 6 Exercise: R-squared for GOOG and VGT

In this exercise, we use monthly CRSP total returns for Google (GOOG) and the Vanguard Information Technology ETF (VGT). We estimate the beta of Google total returns relative to VGT total returns and decompose Google variance into explained and unexplained components.

## Learning objectives

By the end of this exercise, you should be able to:

- Load monthly security returns from CRSP with farms.load_crsp_data().
- Calculate total-return sample variances.
- Estimate a single-factor beta with farms.linear_regression_summary().
- Calculate R-squared as $\beta^2 \operatorname{Var}(VGT)/\operatorname{Var}(GOOG)$ and calculate the unexplained variance share as $1-R^2$.

## Data access

The CRSP examples require a WRDS account with access to CRSP. The connection cell will prompt for WRDS credentials and authentication. Do not place credentials in the notebook. This exercise uses monthly observations and a monthly PeriodIndex.

In [ ]:
%pip install -q --upgrade farms wrds

import farms as fm
import pandas as pd
import wrds

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_rows', 20)

## Establish a WRDS connection

farms.load_crsp_data() accepts an open wrds.Connection object. We identify both securities by ticker and request the inclusive period from January 2021 through December 2025.

In [ ]:
db = wrds.Connection()

start_date = '2021-01'
end_date = '2025-12'

## Load GOOG and VGT returns from CRSP

The CRSP ret field is a decimal total return. We request the total returns for both securities from CRSP.

In [ ]:
crsp = fm.load_crsp_data(
    db=db,
    identifiers=['GOOG', 'VGT'],
    identifier_type='ticker',
    start_date=start_date,
    end_date=end_date,
    frequency='monthly',
    include_factors=None,
)

crsp.head()

## Organize the security returns by date

CRSP returns are initially in long form. We pivot them into one row per month so that GOOG and VGT can be compared using their monthly PeriodIndex.

In [ ]:
security_returns = (
    crsp.reset_index()
    .pivot_table(index='date', columns='ticker', values='ret', aggfunc='first')
    .rename(columns={'GOOG': 'GOOG_return', 'VGT': 'VGT_return'})
    .sort_index()
)

security_returns.head()

## Align total returns

We keep only months for which both securities are available. The regression will use Google total return as the dependent variable and VGT total return as the explanatory variable. This code isn't neded unless you change the date range.

In [ ]:
returns = security_returns.dropna().copy()

returns[['GOOG_return', 'VGT_return']]

## Estimate the variation of the total returns

We use pandas' sample variance, which has ddof=1, for both total-return series. Using the same variance convention for the numerator and denominator is important for the requested R-squared calculation.

In [ ]:
variance = returns[['GOOG_return', 'VGT_return']].var().rename('sample_variance')

variance.to_frame()

## Estimate beta with the farms regression function

The single-factor regression equation is

$$GOOG_t = \alpha + \beta VGT_t + \varepsilon_t.$$

farms.linear_regression_summary() adds the intercept, estimates the coefficients, and returns the coefficient table with confidence intervals. The beta is the coefficient on VGT_return.

In [ ]:
regression = fm.linear_regression_summary(
    X=returns[['VGT_return']],
    Y=returns['GOOG_return'],
)

regression

In [ ]:
beta = regression.loc['VGT_return', 'coef']
alpha = regression.loc['const', 'coef']

pd.Series({'alpha': alpha, 'beta': beta}, name='estimate')

## Calculate R-squared and unexplained variance

For a regression with one explanatory variable and an intercept, the requested variance decomposition is

$$R^2 = \frac{\beta^2 \operatorname{Var}(VGT)}{\operatorname{Var}(GOOG)}.$$

The unexplained variance share is $1-R^2$. The following cell reports both the explained variance component and the requested shares.

In [ ]:
google_variance = variance['GOOG_return']
vgt_variance = variance['VGT_return']
explained_variance = beta**2 * vgt_variance
r_squared = explained_variance / google_variance
unexplained_variance = 1 - r_squared

variance_decomposition = pd.Series({
    'GOOG total-return variance': google_variance,
    'VGT total-return variance': vgt_variance,
    'Beta': beta,
    'Explained variance': explained_variance,
    'R-squared': r_squared,
    'Unexplained variance (1 - R-squared)': unexplained_variance,
}, name='value')

variance_decomposition.to_frame()

## Close the WRDS connection

Close the database connection after the CRSP query is complete.

In [ ]:
db.close()

## Interpretation

The R-squared value is the fraction of the sample variance of GOOG total returns explained by the fitted linear relationship with VGT total returns. The unexplained variance value is the remaining fraction. Both are sample estimates for the January 2021--December 2025 period, not population parameters.